# 03 — Loading the SQLite Database

Loads the cleaned tables from `data/processed/` into a normalized SQLite database at `db/spotify.db`, applying the schema in `sql/schema.sql`. The genre and artist text labels are replaced with surrogate integer keys and their own dimension tables.

## Read the cleaned tables

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

PROC = Path("../data/processed")
tracks = pd.read_csv(PROC / "tracks.csv")
audio_features = pd.read_csv(PROC / "audio_features.csv")
track_genres_raw = pd.read_csv(PROC / "track_genres.csv")
track_artists_raw = pd.read_csv(PROC / "track_artists.csv")

## Build dimension tables with surrogate keys

Each distinct genre / artist name gets a stable integer id. We then translate the bridge tables from names to ids, so the database joins on cheap integers instead of repeated strings.

In [ ]:
genres = pd.DataFrame({"genre_name": sorted(track_genres_raw["track_genre"].unique())})
genres["genre_id"] = range(1, len(genres) + 1)

artists = pd.DataFrame({"artist_name": sorted(track_artists_raw["artist_name"].unique())})
artists["artist_id"] = range(1, len(artists) + 1)

track_genres = (track_genres_raw
    .merge(genres, left_on="track_genre", right_on="genre_name")
    [["track_id", "genre_id"]])

track_artists = (track_artists_raw
    .merge(artists, on="artist_name")
    [["track_id", "artist_id", "position"]])

genres = genres[["genre_id", "genre_name"]]
artists = artists[["artist_id", "artist_name"]]

print(f"genres:  {len(genres):,}")
print(f"artists: {len(artists):,}")

## Create the schema and load the tables

`executescript` runs `schema.sql` (which drops and recreates every table), then each DataFrame is appended into its table. Loading parents before children keeps foreign keys valid.

In [ ]:
DB = Path("../db/spotify.db")
DB.parent.mkdir(exist_ok=True)

conn = sqlite3.connect(DB)
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(Path("../sql/schema.sql").read_text())

for name, frame in [
    ("tracks", tracks),
    ("audio_features", audio_features),
    ("genres", genres),
    ("artists", artists),
    ("track_genres", track_genres),
    ("track_artists", track_artists),
]:
    frame.to_sql(name, conn, if_exists="append", index=False)

conn.commit()
print("loaded")

## Verify the load

Row counts per table, a foreign-key integrity check, and a sample join answering "the 5 most popular tracks and their genres".

In [ ]:
for t in ["tracks", "audio_features", "genres", "artists", "track_genres", "track_artists"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:16s} {n:>8,}")

violations = conn.execute("PRAGMA foreign_key_check").fetchall()
print("\nforeign-key violations:", len(violations))

In [ ]:
pd.read_sql_query(
    """
    SELECT t.track_name, t.popularity, g.genre_name
    FROM tracks t
    JOIN track_genres tg ON tg.track_id = t.track_id
    JOIN genres g        ON g.genre_id  = tg.genre_id
    ORDER BY t.popularity DESC
    LIMIT 5
    """,
    conn,
)